# Notebook 01 — Analyse Exploratoire et Prétraitement (EDA)
## Système de Recommandation Hybride Emploi-Compétences · Cameroun
**NGOULOU-NGOUBILI Irch Defluviaire · ISE M2 · Data Science & Marketing**

---
### Plan du notebook
1. Chargement et audit des données brutes
2. Analyse des offres d'emploi
3. Analyse des demandeurs d'emploi
4. Analyse des référentiels (MEPC, NCF, ESCO)
5. Exécution du pipeline ETL
6. Validation des données traitées et analyse des paires fine-tuning
7. Visualisations synthétiques


## 1. Chargement et audit des données brutes

In [ ]:
import sys, os, json, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

# Chemins projet
ROOT      = Path('..').resolve()
DATA_RAW  = ROOT / 'data' / 'raw'
DATA_PROC = ROOT / 'data' / 'processed'
DATA_FT   = ROOT / 'data' / 'finetune'
SRC_ETL   = ROOT / 'src' / '01_etl'
sys.path.insert(0, str(SRC_ETL))

print("ROOT    :", ROOT)
print("DATA_RAW:", DATA_RAW)
print("Fichiers disponibles:")
for f in sorted(DATA_RAW.iterdir()): print("  ", f.name)


In [ ]:
df_offres = pd.read_excel(DATA_RAW / 'offres.xlsx', dtype=str)
df_cand   = pd.read_excel(DATA_RAW / 'demandeur.xlsx')
mepc_raw  = pd.read_excel(DATA_RAW / 'MEPC_Nomenclature_Camerounaise_Metiers.xlsx', sheet_name=None)
ncf_raw   = pd.read_excel(DATA_RAW / 'NCF_Nomenclature_Camerounaise_Formations.xlsx', sheet_name=None)

print(f"Offres    : {df_offres.shape[0]:,} lignes × {df_offres.shape[1]} colonnes")
print(f"Candidats : {df_cand.shape[0]:,} lignes × {df_cand.shape[1]} colonnes")
print(f"MEPC      : {list(mepc_raw.keys())}")
print(f"NCF       : {list(ncf_raw.keys())}")


## 2. Analyse des offres d'emploi

In [ ]:
# Rapport de qualité (taux de remplissage)
rapport_offres = pd.DataFrame({
    'Colonne': df_offres.columns,
    'Non nuls': [df_offres[c].notna().sum() for c in df_offres.columns],
    'Remplissage (%)': [round(df_offres[c].notna().mean()*100, 1) for c in df_offres.columns],
    'Uniques': [df_offres[c].nunique() for c in df_offres.columns],
})
rapport_offres.style.background_gradient(subset=['Remplissage (%)'], cmap='RdYlGn')


In [ ]:
# Doublons
n_dup = df_offres.duplicated().sum()
print(f"Doublons exacts     : {n_dup:,} ({n_dup/len(df_offres)*100:.1f}%)")
print(f"Après dédoublonnage : {len(df_offres) - n_dup:,} offres uniques")

# Champ Détails de l'Annonce — bruit
details_ok = df_offres["Détails de l'Annonce"].notna()
print(f"\nDétails de l'Annonce remplis : {details_ok.sum():,} ({details_ok.mean()*100:.1f}%)")
print(f"Longueur moyenne (chars)     : {df_offres['Détails de l\'Annonce'].dropna().str.len().mean():.0f}")


In [ ]:
# Distribution par Source
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Groupe de Contrat
gc = df_offres['Groupe de Contrat'].value_counts()
axes[0].barh(gc.index, gc.values, color='#028090')
axes[0].set_title("Groupe de Contrat", fontweight='bold')
axes[0].set_xlabel("Nombre d'offres")

# Niveau d'Études
ne = df_offres["Niveau d'Études"].value_counts()
axes[1].barh(ne.index, ne.values, color='#1E2761')
axes[1].set_title("Niveau d'Études requis", fontweight='bold')
axes[1].set_xlabel("Nombre d'offres")

# Niveau d'Expérience
nexp = df_offres["Niveau d'Expérience"].value_counts()
axes[2].barh(nexp.index, nexp.values, color='#E67E22')
axes[2].set_title("Niveau d'Expérience requis", fontweight='bold')
axes[2].set_xlabel("Nombre d'offres")

plt.suptitle("Distribution des offres d'emploi (données brutes)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Top secteurs d'activité (multi-valeurs explosées)
secteurs = (df_offres["Secteur d'Activité"]
            .str.split(',').explode()
            .str.strip()
            .value_counts()
            .head(20))

fig, ax = plt.subplots(figsize=(10, 6))
secteurs.sort_values().plot(kind='barh', ax=ax, color='#028090', edgecolor='white')
ax.set_title("Top 20 secteurs d'activité dans les offres", fontweight='bold')
ax.set_xlabel("Nombre d'occurrences")
plt.tight_layout()
plt.show()


In [ ]:
# Top villes de recrutement
villes = (df_offres['Ville / Région']
          .str.split(',').explode()
          .str.strip()
          .str.upper()
          .value_counts()
          .head(12))

fig, ax = plt.subplots(figsize=(9, 5))
villes.sort_values().plot(kind='barh', ax=ax, color='#1E2761', edgecolor='white')
ax.set_title("Top 12 villes de recrutement", fontweight='bold')
ax.set_xlabel("Nombre d'offres")
plt.tight_layout()
plt.show()


## 3. Analyse des demandeurs d'emploi

In [ ]:
rapport_cand = pd.DataFrame({
    'Colonne': df_cand.columns,
    'Non nuls': [df_cand[c].notna().sum() for c in df_cand.columns],
    'Remplissage (%)': [round(df_cand[c].notna().mean()*100, 1) for c in df_cand.columns],
    'Uniques': [df_cand[c].nunique() for c in df_cand.columns],
})
rapport_cand.style.background_gradient(subset=['Remplissage (%)'], cmap='RdYlGn')


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Niveau études
ne = df_cand['niveau_etude'].value_counts()
axes[0,0].pie(ne.values, labels=ne.index, autopct='%1.1f%%',
              colors=plt.cm.Set3.colors[:len(ne)])
axes[0,0].set_title("Niveau d'études des candidats", fontweight='bold')

# Genre
g = df_cand['Genre'].value_counts()
axes[0,1].pie(g.values, labels=g.index, autopct='%1.1f%%',
              colors=['#1E2761','#F96167'])
axes[0,1].set_title("Genre", fontweight='bold')

# Secteur métier (top 10)
sm = df_cand['secteur_metier'].value_counts().head(10)
axes[1,0].barh(sm.index, sm.values, color='#028090')
axes[1,0].set_title("Top 10 secteurs métier (candidats)", fontweight='bold')

# Distribution âge
df_cand['Age'].plot(kind='hist', bins=20, ax=axes[1,1],
                    color='#E67E22', edgecolor='white', rwidth=0.85)
axes[1,1].set_title("Distribution des âges", fontweight='bold')
axes[1,1].set_xlabel("Âge")
axes[1,1].set_ylabel("Nombre de candidats")

plt.suptitle("Profil des demandeurs d'emploi (données brutes)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Mapping niveau_etude → code NCF
from config import NIVEAU_ETUDES_CAND_TO_NCF

print("Mapping niveau_etude candidats → code NCF (1-9) :")
for val, code in NIVEAU_ETUDES_CAND_TO_NCF.items():
    n = (df_cand['niveau_etude'] == val).sum()
    print(f"  NCF-{code}  {val:<35} → {n} candidats")

print()
print("Valeurs Mobilité géographique :")
print(df_cand['Mobilité géographique'].value_counts().to_dict())
print(f"  → 91.7% 'Non déclaré' → seront mis à NULL dans le graphe")


## 4. Analyse des référentiels MEPC, NCF, ESCO

In [ ]:
print("=== MEPC — Structure hiérarchique ===")
for sheet, df in mepc_raw.items():
    print(f"  [{sheet}] : {len(df)} entrées | colonnes: {list(df.columns)}")

print()
print("=== NCF — Structure hiérarchique ===")
for sheet, df in ncf_raw.items():
    print(f"  [{sheet}] : {len(df)} entrées | colonnes: {list(df.columns)}")


In [ ]:
# Visualisation hiérarchie MEPC
mepc_grands = mepc_raw['Grands Groupes']
mepc_sous   = mepc_raw['Sous-Groupes']
mepc_base   = mepc_raw['Groupes de Base']

# Nombre de sous-groupes par grand groupe
n_sous = mepc_sous.groupby('Code Grand Groupe').size().reset_index(name='N_sous_groupes')
n_base = mepc_base.groupby('Code Grand Groupe').size().reset_index(name='N_groupes_base')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# N sous-groupes
labels_g = [f"{r['Code']} — {r['Intitulé'][:25]}" for _, r in mepc_grands.iterrows()]
axes[0].barh(labels_g, n_sous['N_sous_groupes'].values, color='#1E2761')
axes[0].set_title("MEPC — Sous-groupes par Grand Groupe", fontweight='bold')
axes[0].set_xlabel("Nombre de sous-groupes")

# N groupes de base
axes[1].barh(labels_g, n_base['N_groupes_base'].values, color='#028090')
axes[1].set_title("MEPC — Groupes de Base par Grand Groupe", fontweight='bold')
axes[1].set_xlabel("Nombre de groupes de base")

plt.suptitle("Structure hiérarchique MEPC 2013 — INS Cameroun", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Pont MEPC ↔ ESCO via ISCO-08
# Analyse des codes CITP disponibles dans la MEPC
codes_citp = (mepc_base['Codes CITP rev.4']
              .dropna()
              .str.replace('/', ',')
              .str.split(',')
              .explode()
              .str.strip()
              .value_counts())
print(f"Codes CITP uniques dans MEPC : {codes_citp.nunique()}")
print(f"Groupes de base MEPC avec codes CITP : {mepc_base['Codes CITP rev.4'].notna().sum()} / {len(mepc_base)}")
print()
print("Extrait :")
print(mepc_base[['Code', 'Intitulé', 'Codes CITP rev.4']].head(8).to_string(index=False))


## 5. Exécution du pipeline ETL

In [ ]:
# Exécution complète du pipeline ETL
import run_etl
results = run_etl.run_all()


## 6. Validation des données traitées

In [ ]:
df_off_proc  = pd.read_parquet(DATA_PROC / 'offres_normalized.parquet')
df_cand_proc = pd.read_parquet(DATA_PROC / 'candidats_normalized.parquet')

print("=== OFFRES NORMALISÉES ===")
print(f"Shape : {df_off_proc.shape}")
print(f"Colonnes : {list(df_off_proc.columns)}")
print()
print("Aperçu NaN par colonne clé :")
key_cols = ['titre_poste', 'ncf_niveau_code', 'secteur_principal',
            'ville_principale', 'experience_min_ans', 'ft_eligible']
for col in key_cols:
    if col in df_off_proc.columns:
        pct_null = df_off_proc[col].isna().mean() * 100
        print(f"  {col:<30} : {pct_null:.1f}% manquants")


In [ ]:
# Distribution NCF dans les offres
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ncf_dist = df_off_proc['ncf_niveau_code'].value_counts().sort_index()
axes[0].bar(ncf_dist.index.astype(str), ncf_dist.values, color='#1E2761', edgecolor='white')
axes[0].set_title("Distribution Code NCF (Offres)", fontweight='bold')
axes[0].set_xlabel("Code NCF (1=Primaire → 9=Doctorat)")
axes[0].set_ylabel("Nombre d'offres")

ncf_cand = df_cand_proc['ncf_niveau_final'].value_counts().sort_index()
axes[1].bar(ncf_cand.index.astype(str), ncf_cand.values, color='#028090', edgecolor='white')
axes[1].set_title("Distribution Code NCF (Candidats)", fontweight='bold')
axes[1].set_xlabel("Code NCF (1=Primaire → 9=Doctorat)")
axes[1].set_ylabel("Nombre de candidats")

plt.suptitle("Alignement Niveaux Études → Codes NCF", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Analyse des paires fine-tuning
with open(DATA_FT / 'pairs_metadata.json') as f:
    meta = json.load(f)

print("=== PAIRES FINE-TUNING SENTENCETRANSFORMER ===")
print(f"Total paires         : {meta['total_paires']:,}")
print(f"Train                : {meta['n_train']:,}")
print(f"Validation           : {meta['n_val']:,}")
print(f"Test                 : {meta['n_test']:,}")
print()
print(f"Longueur moy sentence1 (metadata) : {meta['stats_longueur']['s1_mean']:.0f} chars")
print(f"Longueur moy sentence2 (corpus)   : {meta['stats_longueur']['s2_mean']:.0f} chars")
print()
print("Exemple paire :")
import json as _json
with open(DATA_FT / 'pairs_train.jsonl') as f:
    ex = _json.loads(f.readline())
print("  sentence1:", ex['sentence1'][:120])
print("  sentence2:", ex['sentence2'][:120])


In [ ]:
# Distribution des paires par secteur
n_pairs_secteur = pd.Series(meta['distribution_secteurs']).sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
n_pairs_secteur.sort_values().plot(kind='barh', ax=ax, color='#028090', edgecolor='white')
ax.set_title("Nombre de paires FT par secteur (Top 15)", fontweight='bold')
ax.set_xlabel("Nombre de paires")
plt.tight_layout()
plt.show()


## 7. Synthèse du preprocessing

In [ ]:
print("=" * 60)
print("SYNTHÈSE DU MODULE ETL")
print("=" * 60)

print()
print("📁 DONNÉES BRUTES")
print(f"  Offres  brutes    : 8 695 lignes")
print(f"  Candidats bruts   : 1 105 lignes")

print()
print("📁 DONNÉES NORMALISÉES")
print(f"  Offres normalisées  : {len(df_off_proc):,} offres")
print(f"  Candidats normalisés: {len(df_cand_proc):,} candidats")

print()
print("📁 PROBLÈMES TRAITÉS")
print(f"  ✓ Doublons supprimés       : 834")
print(f"  ✓ Boilerplate scraping nettoyé")
print(f"  ✓ Multi-valeurs explosées  : villes, secteurs, compétences")
print(f"  ✓ Mapping Niveau Études → NCF (1-9)")
print(f"  ✓ Mapping Diplôme → NCF (résolution par priorité)")
print(f"  ✓ 'Non déclaré' → NULL")
print(f"  ✓ Mobilité géographique normalisée")
print(f"  ✓ Types de contrat harmonisés")
print(f"  ✓ Mapping MEPC ↔ ESCO via ISCO-08 (19 980 correspondances)")

print()
print("📁 PAIRES FINE-TUNING ST")
print(f"  Train : {meta['n_train']:,} paires (70%)")
print(f"  Val   : {meta['n_val']:,} paires (15%)")
print(f"  Test  : {meta['n_test']:,} paires (15%)")
print(f"  Format : JSONL — InputExample sentence-transformers")
print(f"  Modèle cible : all-MiniLM-L6-v2")
print(f"  Perte : MultipleNegativesRankingLoss")

print()
print("→ Prochaine étape : src/02_finetune_st/train_sentence_transformer.py")
